# PepperAI — Yield Prediction Model (XGBoost + SHAP)### Local version — Anaconda + RTX GPU (GPU not required for this phase, XGBoost runs fine on CPU)**Before running:**1. Place `Kerala.xls` and `Karnataka.xls` in the same folder as this notebook (e.g. `D:\PepperAI\yield_data\`)2. Update the `YIELD_DATA_DIR` path in Step 1 below3. Run cells top to bottom (Shift+Enter)

## Step 1 — Load Kerala + Karnataka data

In [1]:
import pandas as pd
import os

YIELD_DATA_DIR = r"D:\PepperAI\yield_data"

def load_state_file(filename, state_name):
    path = os.path.join(YIELD_DATA_DIR, filename)
    tables = pd.read_html(path)
    df = tables[0]
    df.columns = ['State', 'District', 'Year', 'Area_ha', 'Production_t', 'Yield_t_ha']
    df['State'] = state_name
    return df

kerala = load_state_file('Kerala.xls', 'Kerala')
karnataka = load_state_file('Karnataka.xls', 'Karnataka')

df = pd.concat([kerala, karnataka], ignore_index=True)
print(f"Kerala: {len(kerala)} rows | Karnataka: {len(karnataka)} rows | Combined: {len(df)} rows")
df.head()

Kerala: 308 rows | Karnataka: 351 rows | Combined: 659 rows


,State,District,Year,Area_ha,Production_t,Yield_t_ha
0,Kerala,1. Alappuzha,2001 - 2002,2054.0,196.0,0.10
1,Kerala,1. Alappuzha,2002 - 2003,1940.0,174.0,0.09
2,Kerala,1. Alappuzha,2003 - 2004,1997.0,167.0,0.08
3,Kerala,1. Alappuzha,2004 - 2005,2079.0,181.0,0.09
4,Kerala,1. Alappuzha,2005 - 2006,2000.0,177.0,0.09


## Step 2 — Clean the data- Strip the "1. " numbering prefix from district names- Parse the Year range (e.g. "2001-02") into a usable numeric year- Drop rows with missing production/yield values

In [3]:
import re

def clean_district(name):
    return re.sub(r'^\d+\.\s*', '', str(name)).strip()

df['District'] = df['District'].apply(clean_district)
df['State'] = df['State'].apply(clean_district)

def parse_year(year_str):
    match = re.match(r'(\d{4})', str(year_str))
    return int(match.group(1)) if match else None

df['Year_num'] = df['Year'].apply(parse_year)

before = len(df)
df = df.dropna(subset=['Production_t', 'Yield_t_ha', 'Area_ha', 'Year_num'])
after = len(df)
print(f"Dropped {before - after} rows with missing values")
print(f"Final dataset: {after} rows")
print(f"Year range: {df['Year_num'].min()} - {df['Year_num'].max()}")
print(f"Districts: {df['District'].nunique()}")
df.head()

Dropped 0 rows with missing values
Final dataset: 659 rows
Year range: 2001 - 2022
Districts: 45


,State,District,Year,Area_ha,Production_t,Yield_t_ha,Year_num
0,Kerala,Alappuzha,2001 - 2002,2054.0,196.0,0.10,2001
1,Kerala,Alappuzha,2002 - 2003,1940.0,174.0,0.09,2002
2,Kerala,Alappuzha,2003 - 2004,1997.0,167.0,0.08,2003
3,Kerala,Alappuzha,2004 - 2005,2079.0,181.0,0.09,2004
4,Kerala,Alappuzha,2005 - 2006,2000.0,177.0,0.09,2005


## Step 3 — Pull matching historical weather (Open-Meteo)For each unique district, we get its approximate coordinates and pull historicalaverage rainfall + temperature for the relevant years.NOTE: this needs an internet connection (only step in this notebook that does).

In [5]:
DISTRICT_COORDS = {
    'Alappuzha': (9.49, 76.33), 'Ernakulam': (10.00, 76.33), 'Idukki': (9.85, 76.97),
    'Kannur': (11.87, 75.37), 'Kasaragod': (12.50, 74.99), 'Kollam': (8.89, 76.61),
    'Kottayam': (9.59, 76.52), 'Kozhikode': (11.25, 75.78), 'Malappuram': (11.07, 76.07),
    'Palakkad': (10.78, 76.65), 'Pathanamthitta': (9.27, 76.79), 'Thiruvananthapuram': (8.52, 76.94),
    'Thrissur': (10.53, 76.21), 'Wayanad': (11.68, 76.13),
    'Kodagu': (12.42, 75.73), 'Chikmagalur': (13.31, 75.77), 'Hassan': (13.01, 76.10),
    'Shimoga': (13.93, 75.57), 'Dakshina Kannada': (12.87, 75.15), 'Udupi': (13.34, 74.75),
    'Uttara Kannada': (14.80, 74.70), 'Chikkamagaluru': (13.31, 75.77),
}

STATE_FALLBACK = {'Kerala': (10.85, 76.27), 'Karnataka': (15.32, 75.71)}

def get_coords(row):
    d = row['District']
    if d in DISTRICT_COORDS:
        return DISTRICT_COORDS[d]
    return STATE_FALLBACK.get(row['State'], (10.85, 76.27))

df['lat'], df['lon'] = zip(*df.apply(get_coords, axis=1))
print("Coordinates assigned. Sample:")
df[['District','State','lat','lon']].drop_duplicates().head(10)

Coordinates assigned. Sample:


,District,State,lat,lon
0,Alappuzha,Kerala,9.49,76.33
22,Ernakulam,Kerala,10.00,76.33
44,Idukki,Kerala,9.85,76.97
66,Kannur,Kerala,11.87,75.37
88,Kasaragod,Kerala,12.50,74.99
110,Kollam,Kerala,8.89,76.61
132,Kottayam,Kerala,9.59,76.52
154,Kozhikode,Kerala,11.25,75.78
176,Malappuram,Kerala,11.07,76.07
198,Palakkad,Kerala,10.78,76.65


In [6]:
import requests
import time

def get_yearly_weather(lat, lon, year):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": f"{year}-01-01", "end_date": f"{year}-12-31",
        "daily": "precipitation_sum,temperature_2m_mean",
        "timezone": "auto"
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        data = r.json()
        rainfall = sum(data['daily']['precipitation_sum'])
        temp = sum(data['daily']['temperature_2m_mean']) / len(data['daily']['temperature_2m_mean'])
        return rainfall, temp
    except Exception:
        return None, None

weather_cache = {}
unique_combos = df[['lat','lon','Year_num']].drop_duplicates()
print(f"Fetching weather for {len(unique_combos)} unique district-year combinations...")
print("(This will take a few minutes due to API rate limits)")

for i, row in enumerate(unique_combos.itertuples()):
    key = (row.lat, row.lon, row.Year_num)
    if key not in weather_cache:
        rainfall, temp = get_yearly_weather(row.lat, row.lon, row.Year_num)
        weather_cache[key] = (rainfall, temp)
        if i % 20 == 0:
            print(f"  {i}/{len(unique_combos)} done...")
        time.sleep(0.2)

print("Weather fetching complete.")

Fetching weather for 440 unique district-year combinations...
(This will take a few minutes due to API rate limits)
  0/440 done...
  20/440 done...
  40/440 done...
  60/440 done...
  80/440 done...
  100/440 done...
  120/440 done...
  140/440 done...
  160/440 done...
  180/440 done...
  200/440 done...
  220/440 done...
  240/440 done...
  260/440 done...
  280/440 done...
  300/440 done...
  320/440 done...
  340/440 done...
  360/440 done...
  380/440 done...
  400/440 done...
  420/440 done...
Weather fetching complete.


In [7]:
df['Rainfall_mm'] = df.apply(lambda r: weather_cache.get((r['lat'], r['lon'], r['Year_num']), (None,None))[0], axis=1)
df['Temp_C'] = df.apply(lambda r: weather_cache.get((r['lat'], r['lon'], r['Year_num']), (None,None))[1], axis=1)

before = len(df)
df = df.dropna(subset=['Rainfall_mm', 'Temp_C'])
print(f"Dropped {before - len(df)} rows where weather fetch failed")
print(f"Final dataset with weather: {len(df)} rows")
df.head()

Dropped 0 rows where weather fetch failed
Final dataset with weather: 659 rows


,State,District,Year,Area_ha,Production_t,Yield_t_ha,Year_num,lat,lon,Rainfall_mm,Temp_C
0,Kerala,Alappuzha,2001 - 2002,2054.0,196.0,0.10,2001,9.49,76.33,2715.9,26.315342
1,Kerala,Alappuzha,2002 - 2003,1940.0,174.0,0.09,2002,9.49,76.33,2559.4,26.516712
2,Kerala,Alappuzha,2003 - 2004,1997.0,167.0,0.08,2003,9.49,76.33,2823.3,26.555068
3,Kerala,Alappuzha,2004 - 2005,2079.0,181.0,0.09,2004,9.49,76.33,3091.8,26.394262
4,Kerala,Alappuzha,2005 - 2006,2000.0,177.0,0.09,2005,9.49,76.33,2892.9,26.530685


## Step 4 — Feature engineeringAdd lag features (previous year's yield for the same district) since past yieldis one of the strongest predictors of future yield.

In [8]:
df = df.sort_values(['District', 'Year_num'])
df['Prev_Yield_t_ha'] = df.groupby('District')['Yield_t_ha'].shift(1)

before = len(df)
df_model = df.dropna(subset=['Prev_Yield_t_ha'])
print(f"Dropped {before - len(df_model)} rows with no previous-year yield (first year per district)")
print(f"Final modeling dataset: {len(df_model)} rows")

df_model['District_code'] = df_model['District'].astype('category').cat.codes
df_model['State_code'] = df_model['State'].astype('category').cat.codes

df_model[['District','State','Year_num','Area_ha','Rainfall_mm','Temp_C','Prev_Yield_t_ha','Yield_t_ha']].head(10)

Dropped 45 rows with no previous-year yield (first year per district)
Final modeling dataset: 614 rows


C:\Users\vojes\AppData\Local\Temp\ipykernel_31064\4086190365.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model['District_code'] = df_model['District'].astype('category').cat.codes
C:\Users\vojes\AppData\Local\Temp\ipykernel_31064\4086190365.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model['State_code'] = df_model['State'].astype('category').cat.codes


,District,State,Year_num,Area_ha,Rainfall_mm,Temp_C,Prev_Yield_t_ha,Yield_t_ha
1,Alappuzha,Kerala,2002,1940.00,2559.4,26.516712,0.10,0.09
2,Alappuzha,Kerala,2003,1997.00,2823.3,26.555068,0.09,0.08
3,Alappuzha,Kerala,2004,2079.00,3091.8,26.394262,0.08,0.09
4,Alappuzha,Kerala,2005,2000.00,2892.9,26.530685,0.09,0.09
5,Alappuzha,Kerala,2006,1872.00,2895.4,26.508219,0.09,0.12
6,Alappuzha,Kerala,2007,1790.00,3262.0,26.496986,0.12,0.12
7,Alappuzha,Kerala,2008,1357.00,2928.4,26.219945,0.12,0.09
8,Alappuzha,Kerala,2009,1387.00,3420.2,26.515342,0.09,0.10
9,Alappuzha,Kerala,2010,1506.15,3494.0,26.638904,0.10,0.10
10,Alappuzha,Kerala,2011,724.76,3376.2,26.421918,0.10,0.19


## Step 5 — Train/test split and model training (XGBoost)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import numpy as np

FEATURES = ['Area_ha', 'Rainfall_mm', 'Temp_C', 'Prev_Yield_t_ha', 'District_code', 'State_code', 'Year_num']
TARGET = 'Yield_t_ha'

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

## Step 6 — Evaluate the model

In [ ]:
preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.4f} tonnes/hectare")
print(f"MAE:  {mae:.4f} tonnes/hectare")
print(f"R²:   {r2:.4f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,6))
plt.scatter(y_test, preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Yield (t/ha)')
plt.ylabel('Predicted Yield (t/ha)')
plt.title('Actual vs Predicted Yield')
plt.tight_layout()
plt.savefig('yield_predictions.png', dpi=120)
plt.show()

## Step 7 — Explainable AI: SHAP feature importanceShows which features drive the model's predictions, and by how much.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120)
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('shap_importance_bar.png', dpi=120)
plt.show()

## Step 8 — Explain a single prediction (waterfall)Example: explain why the model predicted a specific yield for one test sample.

In [ ]:
sample_idx = 0
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[sample_idx],
    X_test.iloc[sample_idx],
    show=False
)
plt.tight_layout()
plt.savefig('shap_waterfall_example.png', dpi=120)
plt.show()

print("Actual yield:", y_test.iloc[sample_idx])
print("Predicted yield:", preds[sample_idx])

## Step 9 — Save the trained modelSaves `yield_model.json` (XGBoost native format) -- you'll need this for the website backend.

In [ ]:
model.save_model('yield_model.json')
print("Model saved to yield_model.json")

district_map = dict(enumerate(df_model['District'].astype('category').cat.categories))
state_map = dict(enumerate(df_model['State'].astype('category').cat.categories))

import json as json_lib
with open('district_encoding.json', 'w') as f:
    json_lib.dump({'district_map': district_map, 'state_map': state_map}, f, indent=2)
print("Encoding maps saved to district_encoding.json")